# TTM Financial Statement Feature Family

Build and audit the five point-in-time TTM families: key metrics, ratios, income statement, cash flow, and balance sheet. The production builders apply the configured filing lag before broadcasting sparse observations onto trading dates.

In [ ]:
import os

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
import django
django.setup()

import pandas as pd
from workflows.features import build_feature_panel_frame_for_symbols

In [ ]:
SYMBOLS = ["AAPL", "MSFT", "NVDA"]
CONFIG = {
    "include_price_technicals": False,
    "include_ta_classic_technicals": False,
    "include_time_calendar_features": False,
    "include_fundamental_change": False,
    "include_statement_quality": False,
    "include_ttm_financial_statements": True,
    "include_event_features": False,
    "include_ownership_features": False,
    "include_economic_indicators": False,
    "include_treasury_rates": False,
    "include_representation_embedding": False,
}

panel, fieldnames, metadata = build_feature_panel_frame_for_symbols(
    symbols=SYMBOLS,
    config=CONFIG,
)
panel.shape, metadata["symbols_processed"]

In [ ]:
TTM_FAMILIES = [
    "key_metrics_ttm",
    "ratios_ttm",
    "income_statement_ttm",
    "cash_flow_ttm",
    "balance_sheet_ttm",
]
family_columns = metadata["feature_family_columns"]
family_summary = pd.DataFrame(
    {
        "family": TTM_FAMILIES,
        "feature_count": [len(family_columns[name]) for name in TTM_FAMILIES],
        "sample_columns": [family_columns[name][:5] for name in TTM_FAMILIES],
    }
)
display(family_summary)
if family_summary["feature_count"].sum() == 0:
    print("No TTM records are loaded. Refresh the five TTM FMP endpoints, then rerun this notebook.")

In [ ]:
ttm_columns = [column for family in TTM_FAMILIES for column in family_columns[family]]
coverage = pd.DataFrame(metadata["coverage_rows"]).query("section_key in @TTM_FAMILIES")
missingness = (
    panel.groupby("symbol")[ttm_columns]
    .apply(lambda frame: frame.notna().mean().mean())
    .rename("mean_non_null_rate")
    .reset_index()
)
display(coverage)
display(missingness)

In [ ]:
preview_columns = [
    column
    for column in [
        "date",
        "symbol",
        "is_ttm__revenue",
        "is_ttm__netincome",
        "cf_ttm__freecashflow",
        "bs_ttm__totalassets",
        "rt_ttm__pricetoearningsratio",
    ]
    if column in panel.columns
]
panel[preview_columns].groupby("symbol", group_keys=False).tail(3)